# kasb-crawler — EXAONE-3.5-7.8B QLoRA 파인튜닝 (Colab)

**목표**: 근거 없는 문단번호를 지어내 인용하는 문제(citation fabrication)를 줄이기 위해, RAFT 스타일로 만든 700개 학습 데이터(근거 있음 600 + 근거없음 거부 100)로 EXAONE-3.5-7.8B-Instruct를 QLoRA(4bit 양자화 상태에서 LoRA만 학습)로 파인튜닝합니다.

**이 노트북의 하이퍼파라미터는 전부 출처가 있습니다.** `[논문 확정값]`은 QLoRA/LoRA 원 논문에 실제로 적힌 숫자이고, `[실무 판단]`은 어느 논문에도 없어서 700개짜리 소규모 데이터셋 특성을 감안해 직접 고른 값입니다(과적합 방지 근거는 학습 중 loss 그래프와 마지막 30문항 평가로 실측 확인 — 논문이 보장해주지 않습니다).

| 하이퍼파라미터 | 이 노트북 값 | 근거 |
|---|---|---|
| 4bit 양자화 방식 | NF4 + Double Quantization, compute dtype bf16 | **[논문 확정값]** QLoRA 논문(Dettmers et al., 2023, arXiv:2305.14314) Section 3, Appendix B.2: "we use NF4 with double quantization and bf16 computation datatype" |
| LoRA rank r | 64 | **[논문 확정값]** 위 논문 Appendix B.2: "We set LoRA r = 64, α = 16" (7B~65B 전 모델 공통) |
| LoRA alpha | 16 | **[논문 확정값]** 위와 동일 문장 |
| LoRA dropout | 0.1 | **[논문 확정값]** 위 논문: 7B/13B급은 0.1, 33B/65B급은 0.05 — EXAONE 7.8B는 7B급이므로 0.1 |
| target_modules | 전체 linear layer (all-linear) | **[논문 확정값]** 위 논문 Section 4, Figure 2: "LoRA on all transformer layers is critical to match 16-bit [full fine-tuning] performance" — attention만 붙이면 성능이 못 미침을 ablation으로 확인 |
| 학습률(LR) | 2e-4, constant 스케줄 | **[논문 확정값]** 위 논문 Table 9 (7B 기준) + Appendix B.2: "we use a constant learning rate schedule" |
| 옵티마이저 | paged_adamw_32bit, adam_beta2=0.999 | **[논문 확정값]** 위 논문 Section 3(paged optimizer) + Appendix B.2(beta2=0.999) |
| max_grad_norm | 0.3 | **[논문 확정값]** 위 논문 Appendix B.2 |
| epoch 수 | 3 (매 epoch 체크포인트 저장) | **[실무 판단]** QLoRA 논문은 epoch이 아니라 step 수만 명시해 700개 규모에 대한 직접 근거가 없음. 참고용으로 LIMA 논문(Zhou et al., 2023)은 1,000개 데이터로 15 epoch을 돌리되 5~10 epoch 사이에서 사람이 직접 품질을 보고 체크포인트를 골랐음(perplexity가 생성 품질과 상관없었다고 명시). 저희는 held-out 30문항 평가가 이미 있으므로 3 epoch을 기본으로 하되 epoch마다 저장해서 실측으로 최적 체크포인트를 고릅니다 — "몇 epoch이 정답"이라는 주장은 어떤 논문에도 없으므로 실측으로 결정합니다. |
| batch size / gradient accumulation | 4 / 4 (실효 배치 16) | **[실무 판단]** 논문 값이 아니라 Colab GPU 메모리 제약에 맞춘 값. OOM 나면 batch를 줄이고 accumulation을 늘려 실효 배치는 유지하세요. |
| weight_decay, warmup | 0 (HF 기본값 그대로) | **[확인 안 됨]** QLoRA 논문 Table 9에 이 값들이 아예 없음. 임의로 숫자를 지어내지 않고 라이브러리 기본값을 그대로 둡니다. |

**중요 — 라이브러리 기본값 함정**: HuggingFace `BitsAndBytesConfig`의 기본값은 `bnb_4bit_quant_type="fp4"`(NF4 아님!), `bnb_4bit_use_double_quant=False`, `bnb_4bit_compute_dtype=torch.float32`입니다. 즉 **아무 설정 없이 4bit만 켜면 QLoRA 논문 설정이 재현되지 않습니다.** 아래 코드에서 전부 명시적으로 지정했으니 이 셀을 지우지 마세요.

## 0. GPU 확인
런타임 > 런타임 유형 변경에서 GPU(A100 권장, 없으면 L4/T4도 가능)를 먼저 켜세요.

In [ ]:
!nvidia-smi

## 1. 라이브러리 설치

In [ ]:
!pip install -q -U "transformers>=5.0,<5.9" accelerate peft bitsandbytes trl datasets huggingface_hub


### 1-1. HuggingFace 다운로드 오류 예방 (Xet 비활성화)
큰 모델(수십 GB)을 받을 때 HuggingFace의 신규 다운로드 방식("Xet")에서 `403 Forbidden: SignatureError: invalid key pair id` 오류나 극도로 느린 다운로드가 실제로 보고되고 있습니다(HuggingFace 서버 쪽 인프라 이슈, huggingface/datasets GitHub 이슈 #8328 등). 환경변수 `HF_HUB_DISABLE_XET`만으로는 특정 버전에서 안 먹히는 버그도 보고돼 있어(huggingface_hub 이슈 #3266), 가장 확실한 방법은 `hf_xet` 패키지 자체를 지워 예전 방식(HTTP) 다운로드로 강제 전환하는 것입니다.

In [ ]:
!pip uninstall -y hf_xet -q
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"  # 버전에 따라 무시될 수 있어 위 uninstall이 실질적인 안전장치

## 2. HuggingFace 로그인
`LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct`가 게이트(라이선스 동의) 모델일 수 있습니다. https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct 에서 먼저 라이선스에 동의한 뒤, 아래 셀에서 토큰(read 권한)으로 로그인하세요.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. 학습 데이터 업로드

**v3: `raft_train_data_v3.jsonl`을 업로드하세요** (`raft_data_gen_v3.py`로 생성 -- v1/v2가 쓴 `raft_train_data_full.jsonl`과는 다른 파일입니다. v3는 grounded/refusal 예시 모두 근거 문서 수를 production의 k=8과 동일하게 맞췄고, 답변에 원문 직접인용을 유도하는 지시가 추가됐습니다).

In [ ]:
from google.colab import files
uploaded = files.upload()  # raft_train_data_v3.jsonl 선택
DATA_PATH = "raft_train_data_v3.jsonl"


### (대안) Google Drive 마운트 — 체크포인트를 세션 종료 후에도 보존하고 싶다면 이 방식 권장

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK_DIR = "/content/drive/MyDrive/kasb_lora"
os.makedirs(WORK_DIR, exist_ok=True)
# raft_train_data_full.jsonl을 미리 이 폴더(WORK_DIR)에 올려뒀다면:
# DATA_PATH = os.path.join(WORK_DIR, "raft_train_data_full.jsonl")

## 4. 데이터셋 로드 및 분할
700건 중 내부 검증(loss 모니터링)용으로 소량만 떼어냅니다. 최종 성능 판단은 이 검증셋이 아니라 **별도로 이미 제외해둔 held-out 30문항**(`exaone_baseline_results.jsonl`)으로만 합니다 — 이 30문항은 애초에 학습 데이터 생성 단계에서부터 빠져 있습니다.

In [ ]:
import json, random
from datasets import Dataset

records = [json.loads(l) for l in open(DATA_PATH, encoding="utf-8") if l.strip()]
print(f"전체 {len(records)}건")

rng = random.Random(42)
rng.shuffle(records)
N_VAL = 40  # 학습 중 loss 모니터링용 내부 검증 (최종 평가 아님)
val_records, train_records = records[:N_VAL], records[N_VAL:]
print(f"train={len(train_records)}  internal_val={len(val_records)}")

train_ds = Dataset.from_list(train_records)
val_ds = Dataset.from_list(val_records)

## 4-1. 학습 데이터 길이 감사 (v1 실패 원인 1: max_length 4096이 최장 예시보다 짧아 정답이 잘렸음)
700건 전체를 실제 채팅 템플릿으로 렌더링해 토큰 길이를 재고, 잘림 0건을 보장하는 `MAX_LENGTH`를 정합니다.


In [ ]:
import math, statistics
from transformers import AutoTokenizer

BASE_MODEL = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

lengths = [
    len(tokenizer.apply_chat_template(r["messages"], tokenize=True, add_generation_prompt=False))
    for r in records
]
ls = sorted(lengths)
pct = lambda p: ls[min(len(ls) - 1, int(len(ls) * p / 100))]
max_len = ls[-1]

print(f"n={len(ls)}  max={max_len}  mean={statistics.mean(ls):.0f}  "
      f"p50={pct(50)}  p95={pct(95)}  p99={pct(99)}")
for th in (4096, 8192, 12288):
    n_over = sum(l > th for l in ls)
    print(f"  {th} 초과: {n_over}건 ({n_over / len(ls) * 100:.1f}%)")

_CANDIDATES = [4096, 6144, 8192, 10240, 12288, 16384, 20480, 24576]
MAX_LENGTH = next((c for c in _CANDIDATES if c >= max_len + 16),
                  math.ceil((max_len + 16) / 1024) * 1024)
assert MAX_LENGTH >= max_len, "MAX_LENGTH < 최장 예시 -- v1 실패 원인(정답 잘림) 재발"
print(f"\n선택된 MAX_LENGTH = {MAX_LENGTH} (최장 {max_len} 토큰을 잘림 없이 포함, 잘리는 예시 0건)")


## 5. 토크나이저 로드 + 채팅 템플릿으로 텍스트 포맷팅
EXAONE 토크나이저에 등록된 공식 chat template(`tokenizer.apply_chat_template`)을 그대로 써서, 모델이 원래 인스트럭션 튜닝될 때 본 형식과 최대한 똑같이 맞춥니다. 템플릿 문자열을 직접 만들어 쓰지 않는 이유: EXAONE의 정확한 특수 토큰(역할 구분자 등)을 추측으로 하드코딩하면 형식이 미묘하게 어긋날 위험이 있기 때문입니다.

In [ ]:
from transformers import AutoTokenizer
from trl import SFTConfig

BASE_MODEL = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

assert "completion_only_loss" in SFTConfig.__dataclass_fields__, \
    "이 trl 버전에는 completion_only_loss가 없음 -- !pip install -U trl 후 런타임 재시작"

_msgs = train_ds[0]["messages"]
assert _msgs[-1]["role"] == "assistant"
_full   = tokenizer.apply_chat_template(_msgs, tokenize=False, add_generation_prompt=False)
_ctx    = tokenizer.apply_chat_template(_msgs[:-1], tokenize=False, add_generation_prompt=False)
_prompt = tokenizer.apply_chat_template(_msgs[:-1], tokenize=False, add_generation_prompt=True)
assert _full.startswith(_prompt), "generation_prompt 렌더링이 전체 렌더링의 접두사가 아님 -- 템플릿 확인 필요"

ASSISTANT_MARKER = _prompt[len(_ctx):]
assert ASSISTANT_MARKER.strip(), "assistant 마커가 비어 있음"
print(f"assistant 턴 시작 마커(실측): {ASSISTANT_MARKER!r}")
print(f"eos_token: {tokenizer.eos_token!r}")
print(f"템플릿 {{% generation %}} 지원: {'{% generation %}' in (tokenizer.chat_template or '')}")

def to_prompt_completion(example):
    msgs = example["messages"]
    assert msgs[-1]["role"] == "assistant", "마지막 메시지가 assistant가 아닌 예시 존재"
    full = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    prompt = tokenizer.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
    assert full.startswith(prompt)
    return {"prompt": prompt, "completion": full[len(prompt):]}

train_ds = train_ds.map(to_prompt_completion, remove_columns=train_ds.column_names)
val_ds = val_ds.map(to_prompt_completion, remove_columns=val_ds.column_names)

# 토큰 경계 검증: 분리 토큰화(prompt+completion)가 전체 토큰화와 완전히 일치해야
# 마스킹 경계가 오염되지 않음 (collator류의 흔한 함정)
for i in range(min(20, len(train_ds))):
    p, c = train_ds[i]["prompt"], train_ds[i]["completion"]
    sep = tokenizer(p, add_special_tokens=False).input_ids + tokenizer(c, add_special_tokens=False).input_ids
    joint = tokenizer(p + c, add_special_tokens=False).input_ids
    assert sep == joint, f"예시 {i}: prompt/completion 경계가 토큰화 시 어긋남 -- 마스킹 경계 오염 위험"
print("경계 검증 통과: 분리 토큰화 == 전체 토큰화 (20/20)")

print("--- completion 예시 (loss가 걸릴 유일한 부분) ---")
print(train_ds[0]["completion"][:400])
print("--- completion 끝부분 (종료 토큰 확인) ---")
print(repr(train_ds[0]["completion"][-40:]))


## 6. 4bit 양자화 설정 (QLoRA 논문 값 그대로)

In [ ]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",             # 기본값 fp4가 아니라 반드시 nf4로 명시 (QLoRA 논문)
    bnb_4bit_use_double_quant=True,        # 기본값 False → True로 명시 (QLoRA 논문)
    bnb_4bit_compute_dtype=torch.bfloat16, # 기본값 float32 → bfloat16으로 명시 (QLoRA 논문)
)

In [ ]:
import glob, re, inspect
from transformers import AutoConfig
from transformers.dynamic_module_utils import get_class_from_dynamic_module
from transformers.masking_utils import create_causal_mask

sig_params = list(inspect.signature(create_causal_mask).parameters.keys())
has_cache_position = "cache_position" in sig_params
print("현재 create_causal_mask 인자:", sig_params)

# 캐시 파일이 아직 없으면(=모델을 한 번도 안 불러온 새 런타임) 가중치는 받지 않고
# 커스텀 모델링 코드 파일만 먼저 받아서 캐시를 채운다.
cfg = AutoConfig.from_pretrained(BASE_MODEL, trust_remote_code=True)
model_class_ref = getattr(cfg, "auto_map", {}).get("AutoModelForCausalLM")
if model_class_ref:
    get_class_from_dynamic_module(model_class_ref, BASE_MODEL)
    print("모델링 코드 캐시 확보:", model_class_ref)

candidates = glob.glob("/root/.cache/huggingface/modules/transformers_modules/**/modeling_exaone.py", recursive=True)
print("찾은 파일:", candidates)

if not candidates:
    print("경고: modeling_exaone.py를 여전히 못 찾음 -- 이 버전은 패치가 불필요하거나 파일명이 다를 수 있음. "
          "패치 없이 다음 셀(모델 로드)로 진행하고, 거기서 에러가 나면 그때 다시 확인하세요.")
else:
    for path in candidates:
        src = open(path, encoding="utf-8").read()
        changed = False

        if "input_embeds=inputs_embeds" in src:
            src = src.replace("input_embeds=inputs_embeds", "inputs_embeds=inputs_embeds")
            changed = True
            print(f"[{path}] input_embeds -> inputs_embeds 치환")

        if not has_cache_position:
            pattern = re.compile(
                r"(attention_mask=attention_mask,\s*\n)(\s*)cache_position=cache_position,\s*\n(\s*)(past_key_values=past_key_values,)"
            )
            new_src, n = pattern.subn(r"\1\3\4", src)
            if n:
                src = new_src
                changed = True
                print(f"[{path}] cache_position 인자 {n}곳 제거(현재 버전 미지원)")

        if changed:
            open(path, "w", encoding="utf-8").write(src)
            print(f"저장 완료: {path}")
        else:
            print(f"[{path}] 변경 없음")


## 7. 베이스 모델 로드 (4bit)

In [ ]:
import time
from transformers import AutoModelForCausalLM

MAX_RETRIES = 30
model = None
for attempt in range(1, MAX_RETRIES + 1):
    try:
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
        print(f"성공! (시도 {attempt}회 만에)")
        break
    except OSError as e:
        print(f"[시도 {attempt}/{MAX_RETRIES}] 실패 -- 30초 후 재시도 ({type(e).__name__})")
        time.sleep(30)

if model is None:
    raise RuntimeError("30회 재시도해도 실패 -- HuggingFace 서버 문제가 아직 안 풀린 것 같습니다.")
model.config.use_cache = False


## 8. LoRA 설정 (QLoRA 논문 값 그대로) + 학습 준비

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch.nn as nn, types

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# EXAONE 커스텀 코드가 get_input_embeddings를 구현 안 해서 생기는 문제 우회:
# 실제 임베딩 레이어(nn.Embedding)를 직접 찾아 get/set_input_embeddings를 붙여준다.
def _find_embedding_module(m):
    for name, module in m.named_modules():
        if isinstance(module, nn.Embedding):
            return name, module
    return None, None

_emb_name, _emb_module = _find_embedding_module(model)
print("찾은 임베딩 레이어:", _emb_name)
assert _emb_name is not None, "임베딩 레이어를 못 찾았습니다 — model 구조를 print(model)로 확인해주세요"

def _get_input_embeddings(self):
    return _emb_module
def _set_input_embeddings(self, value):
    parent = self
    *path, last = _emb_name.split(".")
    for p in path:
        parent = getattr(parent, p)
    setattr(parent, last, value)

model.get_input_embeddings = types.MethodType(_get_input_embeddings, model)
model.set_input_embeddings = types.MethodType(_set_input_embeddings, model)

lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


`trainable params` 비율이 몇 %인지 여기서 꼭 확인하세요(QLoRA 7B 기준으로 대략 전체 파라미터의 1~3% 수준이 나오면 정상입니다 — r=64/all-linear 조합에서 크게 벗어나면 target_modules가 EXAONE 아키텍처에서 의도대로 안 잡혔을 수 있으니 `model.print_trainable_parameters()` 출력과 함께 확인 부탁드립니다).

## 9. 학습 설정 (SFTConfig)

In [ ]:
from trl import SFTConfig

OUTPUT_DIR = "/content/drive/MyDrive/kasb_lora/checkpoints" if 'WORK_DIR' in dir() else "./checkpoints"

if MAX_LENGTH > 8192:
    BS, GA = 1, 16
elif MAX_LENGTH > 4096:
    BS, GA = 2, 8
else:
    BS, GA = 4, 4
print(f"MAX_LENGTH={MAX_LENGTH} -> batch={BS}, grad_accum={GA} (실효 배치 {BS*GA})")

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=BS,
    gradient_accumulation_steps=GA,
    learning_rate=2e-4,                  # [논문 확정값] QLoRA Table 9 (7B)
    lr_scheduler_type="constant",        # [논문 확정값] QLoRA Appendix B.2
    optim="paged_adamw_32bit",           # [논문 확정값] QLoRA paged optimizer
    adam_beta2=0.999,                    # [논문 확정값] QLoRA Appendix B.2
    max_grad_norm=0.3,                   # [논문 확정값] QLoRA Appendix B.2
    bf16=True,
    logging_steps=10,
    eval_strategy="epoch",               # eval_loss는 계속 로깅(참고/진단용) -- 선택 기준으로는 안 씀
    save_strategy="epoch",
    save_total_limit=3,                  # epoch 3개 어댑터 체크포인트 전부 Drive 보존 (각 수백MB)
    # v3: load_best_model_at_end / metric_for_best_model="eval_loss" 제거.
    # v2가 이 기준으로 골랐다가 Faithfulness가 v1보다도 나빠졌음(0.750) -- LIMA 논문
    # (Zhou et al. 2023, arXiv:2305.11206)이 "perplexity는 생성 품질과 무상관"이라 경고한
    # 그대로였음. 대신 아래 11번 셀에서 매 epoch을 실제로 생성시켜 판사로 채점해 고른다.
    max_length=MAX_LENGTH,               # 감사 셀 실측 -- 잘림 0건 보장
    packing=False,
    completion_only_loss=True,           # prompt(근거문서+질문) loss 제외, completion(정답)에만 학습
    report_to="none",
)


## 10. 학습 실행

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

# --- 학습 전 마스킹 검증: 실제 collate 결과에서 loss 대상 토큰 확인 ---
_rows = [trainer.train_dataset[i] for i in range(2)]
_batch = trainer.data_collator(_rows)
_labels = _batch["labels"]
_norm = lambda s: "".join(s.split())
for _i in range(_labels.shape[0]):
    lab = _labels[_i]
    n_total = int(_batch["attention_mask"][_i].sum()) if "attention_mask" in _batch else lab.numel()
    n_train = int((lab != -100).sum())
    assert n_train > 0, f"예시 {_i}: loss 대상 토큰 0개 -- 전체가 마스킹됨(마스킹 실패)"
    assert n_train < n_total, f"예시 {_i}: 전체 토큰에 loss -- 마스킹 미적용"
    print(f"예시 {_i}: 전체 {n_total} 토큰 중 loss 대상 {n_train}개 ({n_train/n_total:.1%})")
    if n_train / n_total > 0.7:
        print(f"  경고: loss 비율이 비정상적으로 높음 -- 아래 디코드 출력을 눈으로 확인할 것")
    trained_text = tokenizer.decode([t for t in lab.tolist() if t != -100])
    expected = train_ds[_i]["completion"]
    assert _norm(expected)[:30] in _norm(trained_text), \
        f"예시 {_i}: loss 대상 토큰이 completion과 불일치 -- 마스킹 경계 오류"
    assert _norm(train_ds[_i]["prompt"])[len(_norm(train_ds[_i]["prompt"]))//2:][:30] not in _norm(trained_text), \
        f"예시 {_i}: prompt(근거문서) 내용에 loss가 걸려 있음"

_recon = tokenizer.decode(_batch["input_ids"][0][_batch["attention_mask"][0].bool()])
_orig = train_ds[0]["prompt"] + train_ds[0]["completion"]
assert _norm(_orig)[:100] in _norm(_recon), "collate된 input_ids가 원본 렌더링과 불일치"
print("입력 끝부분(종료 토큰 중복 여부 확인):", repr(_recon[-80:]))
print("--- loss 대상 디코드(앞 200자) -- 정답 답변이어야 함 ---")
print(trained_text[:200])
print("\n마스킹 검증 통과 -- 학습 시작")

trainer.train()


학습 로그의 `loss`(train)와 `eval_loss`(내부 검증)를 epoch마다 비교해서 눈으로만 참고하세요(과적합 조짐 파악용). **최종 체크포인트 선택은 이 값으로 하지 않습니다** -- 바로 다음 셀에서 매 epoch을 실제로 생성시켜 판사로 채점하는 방식으로 자동 선택합니다 (LIMA 논문이 지적한, perplexity와 실제 생성 품질의 무상관 문제를 피하기 위함).

## 11. 체크포인트별 생성 + 판사 채점 -> 최적 epoch 자동 선택

v2는 `eval_loss`가 가장 낮은 체크포인트를 자동 선택했는데, 그 체크포인트가 실제로는 가장 나빴습니다(Faithfulness 0.750, v1의 0.800보다도 낮음). LIMA 논문이 경고한 "perplexity는 생성 품질과 상관없다"가 그대로 재현된 것으로 보입니다.

그래서 v3는 매 epoch 체크포인트마다 **internal-val 40개에 대해 실제로 답변을 생성**시키고, **GPT-4o-mini 판사**(지금까지 베이스라인/v1/v2 채점과 동일 모델 -- 비교 일관성 유지)로 채점해서, 그 점수가 가장 높은 체크포인트를 고릅니다. 채점 기준: grounded 문항은 근거에 실제로 부합하게 답했으면 1점(지어내면 0점, 부당 거부하면 0점), refusal 문항은 실제로 거부했으면 1점(근거 없이 지어내 답하면 0점).

OpenAI API 키가 필요합니다(판사 호출용, Colab 세션에만 보관되고 저장되지 않습니다).

In [ ]:
import gc, glob, json, os, re
from getpass import getpass

import httpx
import torch
from openai import OpenAI as _OpenAI

OPENAI_API_KEY = getpass("OpenAI API 키 입력 (판사용, 세션에만 보관, 저장 안 됨): ").strip()
_judge_client = _OpenAI(api_key=OPENAI_API_KEY, timeout=httpx.Timeout(120.0, connect=10.0))
JUDGE_MODEL = "gpt-4o-mini"   # 베이스라인/v1/v2 채점과 동일 모델 -- 비교 일관성 유지

# 키 사전 검증 -- 틀린 키로 40건x3체크포인트를 다 돌리고 나서야 알아채는 것을 방지
try:
    _judge_client.chat.completions.create(
        model=JUDGE_MODEL, temperature=0, max_tokens=5,
        messages=[{"role": "user", "content": "ping"}])
    print("OpenAI API 키 확인 완료 -- 판사 호출 가능")
except Exception as e:  # noqa: BLE001
    raise RuntimeError(
        f"OpenAI API 키 검증 실패: {e}\n"
        "-- 키를 다시 복사(앞뒤 공백/줄바꿈 없이)해서 이 셀을 재실행하세요. "
        "https://platform.openai.com/account/api-keys 에서 키가 살아있는지도 확인하세요.") from e

JUDGE_SYS = (
    "너는 한국 회계기준 RAG 답변 채점자다. 질문/근거/모델답변/정답범주를 보고 JSON으로만 "
    '답한다: {"score": 0~1 실수, "reason": "한 문장"}. 채점 기준: '
    "정답범주가 'grounded'면 -- 모델이 실제로 답변했고(거부하지 않았고) 그 내용이 근거에 "
    "있는 사실과 일치하면 1.0, 근거에 없는 내용을 지어냈으면 0.0, 일부만 지어냈으면 0.5, "
    "정당한 근거가 있는데도 거부했으면(부당 거부) 0.0. "
    "정답범주가 'refusal'이면 -- 모델이 실제로 거부했으면(짧게 모른다고 답했으면) 1.0, "
    "근거에 없는 내용을 지어내 답했으면 0.0."
)


def judge_one(question, grounding, category, model_answer):
    user = (f"정답범주: {category}\n\n질문: {question}\n\n근거:\n{grounding[:3000]}\n\n"
            f"모델 답변: {model_answer}")
    try:
        r = _judge_client.chat.completions.create(
            model=JUDGE_MODEL, temperature=0, response_format={"type": "json_object"},
            messages=[{"role": "system", "content": JUDGE_SYS}, {"role": "user", "content": user}])
        return float(json.loads(r.choices[0].message.content).get("score", 0.0))
    except Exception as e:  # noqa: BLE001
        print("  [판사 오류]", e)
        return None


def generate_answer(gen_model, prompt_text, max_new_tokens=512):
    inputs = tokenizer(prompt_text, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        out = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                 repetition_penalty=1.15, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


# 학습이 끝났으니 옵티마이저 상태 등 학습 전용 메모리를 먼저 비운다 (생성 메모리 확보)
del trainer
gc.collect()
torch.cuda.empty_cache()

ckpt_dirs = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"),
                   key=lambda p: int(re.search(r"checkpoint-(\d+)", p).group(1)))
assert ckpt_dirs, f"{OUTPUT_DIR}에 checkpoint-* 없음 -- save_strategy 설정을 확인하세요"
print(f"평가할 체크포인트: {[os.path.basename(c) for c in ckpt_dirs]} ({len(val_records)}건씩 채점)")

model.eval()
results = {}
for ckpt in ckpt_dirs:
    name = os.path.basename(ckpt)
    print(f"\n=== {name} 로드 + 생성 + 채점 ===")
    model.load_adapter(ckpt, adapter_name=name)   # 기존 PeftModel에 이 체크포인트를 추가 로드
    model.set_adapter(name)                       # 생성 시 이 어댑터를 활성화

    scores = []
    for i, rec in enumerate(val_records):
        msgs = rec["messages"]
        prompt_text = tokenizer.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
        user_content = msgs[1]["content"]   # "질문: ...\n\n근거:\n..."
        question = user_content.split("근거:")[0].replace("질문:", "").strip()
        grounding = user_content.split("근거:", 1)[1] if "근거:" in user_content else user_content
        ans = generate_answer(model, prompt_text)
        sc = judge_one(question, grounding, rec["category"], ans)
        if sc is not None:
            scores.append(sc)
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(val_records)} 진행, 현재 평균 {sum(scores)/len(scores):.3f}")

    avg = sum(scores) / len(scores) if scores else 0.0
    results[ckpt] = avg
    print(f"{name} 최종 평균 점수: {avg:.3f} ({len(scores)}/{len(val_records)}건 채점됨)")

best_ckpt = max(results, key=results.get)
print("\n=== 체크포인트별 점수 (생성+판사 채점 기준) ===")
for c, s in sorted(results.items(), key=lambda x: -x[1]):
    marker = " <- 선택" if c == best_ckpt else ""
    print(f"  {os.path.basename(c)}: {s:.3f}{marker}")
print(f"\n최종 선택: {best_ckpt}")


In [ ]:
import shutil

ADAPTER_DIR = f"{OUTPUT_DIR}/best_adapter"
if os.path.exists(ADAPTER_DIR):
    shutil.rmtree(ADAPTER_DIR)
shutil.copytree(best_ckpt, ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("저장 완료 (생성+판사 채점 기준 최적 체크포인트):", ADAPTER_DIR, "<-", os.path.basename(best_ckpt))
print("나머지 epoch 체크포인트도 Drive에 보존됨:", f"{OUTPUT_DIR}/checkpoint-*")


## 12. 병합(merge) — LoRA 어댑터를 원본 가중치에 합치기

학습은 4bit로 했지만, 배포용 가중치를 만들려면 **양자화 안 된 상태(bf16)로 베이스 모델을 다시 로드**해서 그 위에 LoRA를 합쳐야 합니다(4bit 가중치에는 직접 병합이 안 됩니다). 이 단계는 GPU 메모리를 더 많이 씁니다(7.8B bf16 ≈ 15.6GB) — A100(40GB) 권장, T4(16GB)는 부족할 수 있습니다.

## 13. GGUF 변환 + 양자화 (Ollama 배포용)

`mlx_lm`의 GGUF 변환은 EXAONE 아키텍처를 지원하지 않아 `llama.cpp`의 `convert_hf_to_gguf.py`를 씁니다. 원본 노트북의 셀 34~52는 디스크 부족으로 여러 번 재시도한 흔적(중복·수동 임기응변 셀)이라 아래 2개 셀로 통합했습니다: **성공한 단계 직후 이전 산출물을 즉시 삭제**하고, HF 캐시(베이스 모델 원본)는 재사용을 위해 보존하며, `load_best_model_at_end`로 뽑힌 best 체크포인트 1개만 변환합니다(60GB 예산, 3회 반복 안 함).


In [ ]:
import os
!pip uninstall -y torchao hf_xet -q
os.environ["HF_HUB_DISABLE_XET"] = "1"

!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!cmake -B llama.cpp/build -S llama.cpp -DCMAKE_BUILD_TYPE=Release
!cmake --build llama.cpp/build --target llama-quantize -j 4
assert os.path.exists("llama.cpp/build/bin/llama-quantize"), "llama-quantize 빌드 실패"


In [ ]:
import os, gc, sys, types, shutil, subprocess
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

def _require_free(gb, step, path="/content"):
    free = shutil.disk_usage(path).free / 1024**3
    print(f"[{step}] 디스크 여유 {free:.1f}GB (필요 최소 {gb}GB)")
    if free < gb:
        raise RuntimeError(
            f"[{step}] 여유 {free:.1f}GB < {gb}GB -- 중단. /content의 불필요 파일을 지우고 "
            f"재실행하세요. 단, HF 캐시(/root/.cache/huggingface)는 지우지 말 것(재사용됨).")

def _rm(path):
    if path and os.path.isdir(path):
        shutil.rmtree(path, ignore_errors=True)
    elif path and os.path.exists(path):
        os.remove(path)

def convert_adapter_to_gguf(adapter_dir, tag, base_model=BASE_MODEL, work="/content"):
    """어댑터 1개 -> 병합(bf16) -> GGUF f16 -> Q4_K_M. 다른 epoch을 시도하려면
    adapter_dir와 tag만 바꿔 재호출. 완성된 중간산출물이 있으면 그 단계는 건너뜀.

    디스크 예산(60GB) 피크: HF캐시 16 + merged 16 + f16 16 ~= 47GB (2단계 변환 구간이 최대).
    참고: convert_hf_to_gguf.py --outtype은 f32/f16/bf16/q8_0/tq1_0/tq2_0만 지원 --
    Q8_0 배포라면 --outtype q8_0으로 f16 중간 단계를 생략할 수 있으나,
    Q4_K_M(K-quant)은 llama-quantize 전용이므로 f16 경유가 필수
    (q8_0에서 재양자화는 --allow-requantize 필요 + 품질 손실).
    """
    merged_dir = f"{work}/merged_{tag}"
    f16_path   = f"{work}/exaone_{tag}_f16.gguf"
    fixed_path = f"{work}/exaone_{tag}_f16_fixed.gguf"
    q4_path    = f"{work}/exaone_{tag}_q4_k_m.gguf"
    in_progress = None
    try:
        # 1/4 병합 -- 피크: 캐시16 + merged16 ~= 32GB
        if not (os.path.exists(merged_dir) or os.path.exists(f16_path) or os.path.exists(fixed_path)):
            _require_free(18, "1/4 병합")
            tok = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
            base = AutoModelForCausalLM.from_pretrained(
                base_model, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
            _emb = next((m for m in base.modules() if isinstance(m, nn.Embedding)), None)
            assert _emb is not None, "EXAONE 임베딩 레이어를 못 찾음"
            base.get_input_embeddings = types.MethodType(lambda self: _emb, base)
            merged = PeftModel.from_pretrained(base, adapter_dir).merge_and_unload()
            in_progress = merged_dir
            merged.save_pretrained(merged_dir, safe_serialization=True)
            tok.save_pretrained(merged_dir)
            in_progress = None
            del base, merged
            gc.collect(); torch.cuda.empty_cache()

        # 2/4 GGUF f16 변환 -- 피크: 캐시16 + merged16 + f16 16 ~= 47GB (전체 최대 피크)
        if not (os.path.exists(f16_path) or os.path.exists(fixed_path)):
            _require_free(18, "2/4 GGUF f16 변환")
            in_progress = f16_path
            subprocess.run(["python", "llama.cpp/convert_hf_to_gguf.py", merged_dir,
                            "--outfile", f16_path, "--outtype", "f16"], check=True)
            assert os.path.getsize(f16_path) > 10 * 1024**3, "f16 gguf가 비정상적으로 작음(불완전)"
            in_progress = None
            _rm(merged_dir)  # 성공 즉시 병합본 삭제 -> +16GB 회수

        # 3/4 EXAONE 메타데이터 보정 -- rms_epsilon 키가 없을 때만 실행
        #     (최신 llama.cpp가 exaone 메타데이터를 제대로 쓰면 자동 스킵됨)
        src_gguf = fixed_path if os.path.exists(fixed_path) else f16_path
        if src_gguf == f16_path:
            sys.path.insert(0, "llama.cpp/gguf-py")
            import gguf
            reader = gguf.GGUFReader(f16_path, "r")
            arch = reader.get_field(gguf.Keys.General.ARCHITECTURE).contents()
            has_rms = reader.get_field(f"{arch}.attention.layer_norm_rms_epsilon") is not None
            eps_field = reader.get_field(f"{arch}.attention.layer_norm_epsilon")
            if not has_rms and eps_field is not None:
                _require_free(18, "3/4 메타데이터 보정")  # f16 두 벌 공존 구간: 캐시16 + 16*2 ~= 47GB
                for _p in ("llama.cpp/gguf-py/gguf/scripts", "llama.cpp/gguf-py/scripts"):
                    if os.path.isdir(_p):
                        sys.path.insert(0, _p)
                try:
                    from gguf_new_metadata import copy_with_new_metadata, MetadataDetails
                except ImportError as e:
                    raise RuntimeError("gguf_new_metadata.py를 찾지 못함 -- llama.cpp 버전 확인 필요") from e
                new_md = {f"{arch}.attention.layer_norm_rms_epsilon":
                          MetadataDetails(gguf.GGUFValueType.FLOAT32, eps_field.contents())}
                in_progress = fixed_path
                writer = gguf.GGUFWriter(fixed_path, arch=arch, endianess=reader.endianess)
                copy_with_new_metadata(reader, writer, new_md, remove_metadata=[])
                in_progress = None
                del reader; gc.collect()
                os.remove(f16_path)
                src_gguf = fixed_path
            else:
                del reader; gc.collect()

        # 4/4 Q4_K_M 양자화 -- 피크: 캐시16 + f16 16 + q4 5 ~= 37GB
        _require_free(7, "4/4 Q4_K_M 양자화")
        in_progress = q4_path
        subprocess.run(["./llama.cpp/build/bin/llama-quantize", src_gguf, q4_path, "Q4_K_M"],
                       check=True)
        assert os.path.getsize(q4_path) > 3 * 1024**3, "q4 gguf가 비정상적으로 작음(불완전)"
        in_progress = None
        os.remove(src_gguf)  # 성공 즉시 f16 삭제 -> +16GB 회수

        print(f"완료: {q4_path} ({os.path.getsize(q4_path)/1024**3:.2f} GB)")
        return q4_path
    except BaseException:
        _rm(in_progress)  # 만들다 만 파일만 삭제 -- 완성된 중간산출물은 보존(재실행 시 이어감)
        raise

# ---- 실행: best 체크포인트 1개만 변환 (3회 반복 안 함 -- 디스크 문제의 핵심 해결) ----
for _v in ("model", "trainer", "base", "merged"):
    if _v in globals():
        del globals()[_v]
gc.collect(); torch.cuda.empty_cache()

ADAPTER_DIR = f"{OUTPUT_DIR}/best_adapter"
q4 = convert_adapter_to_gguf(ADAPTER_DIR, tag="best")
dst = os.path.join(OUTPUT_DIR, os.path.basename(q4))
shutil.copy(q4, dst)
print("Drive 저장 완료:", dst)

# held-out 30문항 평가에서 best가 실패하면, 다른 epoch을 어댑터 경로만 바꿔 재시도:
# convert_adapter_to_gguf(f"{OUTPUT_DIR}/checkpoint-<step>", tag="ep2")


## 14. (로컬 맥에서 실행) Ollama에 새 태그로 배포

**아래는 Colab이 아니라 로컬 맥 터미널에서 실행하는 명령입니다.** 기존 태그들은 롤백/비교용으로
그대로 남겨두고 새 태그로 추가합니다.
- `exaone3.5:7.8b`(베이스, 거부율 33%/Faithfulness 0.900 -- 지금까지 최고 성능)
- `exaone3.5-lora:v1`(1차 재학습 실패, 거부율 66.7%/Faithfulness 0.800)
- `exaone3.5-lora:v2`(2차 재학습 실패, 거부율 46.7%/Faithfulness 0.750 -- eval_loss 기준 체크포인트 선택의 부작용으로 추정)

```bash
# 1) 기존 태그의 template/parameter를 그대로 재사용하기 위해 먼저 확인
ollama show exaone3.5:7.8b --modelfile > /tmp/exaone_base.modelfile
cat /tmp/exaone_base.modelfile

# 2) 새 Modelfile 작성 -- FROM 줄만 새 gguf로 바꾸고 TEMPLATE/SYSTEM은 위에서 확인한 걸 그대로 복사.
#    PARAMETER 블록에는 repeat_penalty/num_predict를 반드시 추가하세요 -- v1이 route() 노드에서
#    반복생성 무한루프에 빠졌던 문제의 재발 방지 안전장치입니다(재학습 여부와 무관하게 항상 넣습니다).
cat > /tmp/exaone_lora_v3.modelfile <<'EOF'
FROM /path/to/exaone_best_q4_k_m.gguf
# (여기에 exaone_base.modelfile의 TEMPLATE 블록을 그대로 붙여넣기)
PARAMETER repeat_penalty 1.15
PARAMETER num_predict 2048
EOF

# 3) 새 태그로 생성 (기존 태그들은 그대로 보존됨)
ollama create exaone3.5-lora:v3 -f /tmp/exaone_lora_v3.modelfile
```


## 15. 필수 -- held-out 30문항 재평가

**이 단계 없이는 재학습이 실제로 도움이 됐는지 알 수 없습니다.** `KASB_LOCAL_MODEL=exaone3.5-lora:v3`
환경변수로 held-out 30문항 평가 스크립트를 다시 돌려서, 지금까지의 세 결과와 모두 비교하세요.

| | 거부율 | Faithfulness |
|---|---|---|
| 베이스라인(파인튜닝 안 함) | 33.0% | **0.900** (지금까지 최고) |
| v1(표준 SFT, eval_loss 선택) | 66.7% | 0.800 |
| v2(completion-only loss, eval_loss 선택) | 46.7% | 0.750 (v1보다도 낮음) |
| v3(위 두 개 + 생성 기반 체크포인트 선택) | ? | ? |

- Faithfulness가 0.900을 넘고 거부율이 과도하게(예: 40%+) 튀지 않으면 -> 성공, 새 태그를 실제로 스왑
- 이번에도 베이스라인을 못 넘으면 -> Drive에 보존된 다른 epoch 체크포인트(`checkpoint-<step>`)를
  `convert_adapter_to_gguf()`로 재변환해 시도해보되, 그래도 안 되면 파인튜닝 자체를 재고할 근거로
  기록하세요(리서치에서 확인된 "According to..." 프롬프팅 등 무학습 대안이 이미 후보로 있습니다).

```bash
KASB_LOCAL_MODEL=exaone3.5-lora:v3 python exaone_v3_final_eval.py
```
